In [0]:
import requests
import json
from datetime import datetime
import time

# API key 
API_KEY = "d6bb76a0-3fdc-4736-b363-23e7e581bc11"

BASE_URL = "https://api.company-information.service.gov.uk/company"

BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/companies_house/"

PATHS = {
    "overview": BASE_PATH + "overview/",
    "filing": BASE_PATH + "filing_history/",
    "people": BASE_PATH + "people/"
}

companies = [
    {"name": "J SAINSBURY PLC", "number": "00185647"},
    {"name": "JD SPORTS FASHION PLC", "number": "01888425"},
    {"name": "OCADO GROUP PLC", "number": "07098618"},
    {"name": "LLOYDS BANKING GROUP PLC", "number": "SC095000"},
    {"name": "NATWEST GROUP PLC", "number": "SC045551"},
    {"name": "NATIONAL GRID PLC", "number": "04031152"},
    {"name": "SSE PLC", "number": "SC117119"},
    {"name": "DRAX GROUP PLC", "number": "05562053"},
    {"name": "BALFOUR BEATTY PLC", "number": "00395826"},
    {"name": "PERSIMMON PLC", "number": "01818486"},
    {"name": "EASYJET PLC", "number": "03959649"},
    {"name": "INTERCONTINENTAL HOTELS GROUP PLC", "number": "05134420"},
    {"name": "HIKMA PHARMACEUTICALS PLC", "number": "05557934"},
    {"name": "PEARSON PLC", "number": "00053723"},
    {"name": "BT GROUP PLC", "number": "04190816"},
    {"name": "ITV PLC", "number": "04967001"},
    {"name": "JOHNSON MATTHEY PLC", "number": "00033774"},
    {"name": "MELROSE INDUSTRIES PLC", "number": "09800044"},
    {"name": "AVIVA PLC", "number": "02468686"},
    {"name": "ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC", "number": "11488166"}
]

# -------------------------------
# Helper function
# -------------------------------
def fetch_and_save(url, path):
    try:
        res = requests.get(url, auth=(API_KEY, ""))
        
        if res.status_code == 200:
            dbutils.fs.put(path, json.dumps(res.json()), overwrite=True)
            return "success"
        else:
            return f"failed ({res.status_code})"
    
    except Exception as e:
        return f"error ({e})"


# -------------------------------
# Main ingestion
# -------------------------------
def ingest_company(company):

    company_number = company["number"]
    company_name = company["name"]
    ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    print(f"\n Processing: {company_name} ({company_number})")

    # Overview
    status = fetch_and_save(
        f"{BASE_URL}/{company_number}",
        f"{PATHS['overview']}{company_number}_{ts}.json"
    )
    print(f"Overview: {status}")

    # Filing History
    status = fetch_and_save(
        f"{BASE_URL}/{company_number}/filing-history",
        f"{PATHS['filing']}{company_number}_{ts}.json"
    )
    print(f"Filing: {status}")

    # People
    status = fetch_and_save(
        f"{BASE_URL}/{company_number}/officers",
        f"{PATHS['people']}{company_number}_{ts}.json"
    )
    print(f"People: {status}")

    # Avoid throttling
    time.sleep(1)


# -------------------------------
# Run
# -------------------------------
for company in companies:
    ingest_company(company)